# Análisis de Sentimiento en Reclamaciones Bancarias

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Analizar sentimiento** de reclamaciones con `CORTEX.SENTIMENT()`
2. **Clasificar reclamaciones** por categoría con `CORTEX.COMPLETE()`
3. **Extraer entidades** y temas clave de textos
4. **Priorizar reclamaciones** por urgencia y sentimiento
5. **Generar respuestas automatizadas** con LLM

---

## Lo Que Construirás

Un **sistema de análisis de reclamaciones** bancarias:
- Scoring de sentimiento automático (-1 a +1)
- Clasificación por categoría (comisiones, servicio, digital, fraude)
- Detección de urgencia y escalado automático
- Respuestas borrador generadas por IA
- Dashboard de monitorización de calidad

**Valor de Negocio**: Mejorar tiempos de respuesta y satisfacción del cliente en reclamaciones.

---

## Funcionalidades Clave

- `CORTEX.SENTIMENT()` — Análisis de sentimiento de textos
- `CORTEX.COMPLETE()` — Clasificación, extracción y generación de respuestas
- `GENERATOR()` — Reclamaciones sintéticas realistas
- `CASE` + `NTILE` — Priorización por urgencia

¡Comencemos!

---

## Paso 1: Configuración del Entorno

### Gestión de Reclamaciones

**Objetivo**: Automatizar el análisis y respuesta de reclamaciones bancarias.

### Categorías de Reclamaciones

| Categoría | Ejemplo | % Aproximado |
|---|---|---|
| Comisiones | Cobros inesperados en cuenta | 35% |
| Servicio | Mala atención en oficina | 25% |
| Digital | Problemas con app/web | 20% |
| Fraude | Cargos no autorizados | 10% |
| Otros | Hipotecas, seguros | 10% |

In [ ]:
CREATE DATABASE IF NOT EXISTS SENTIMIENTO_RECLAM_DB;
CREATE SCHEMA IF NOT EXISTS SENTIMIENTO_RECLAM_DB.PUBLIC;
USE SCHEMA SENTIMIENTO_RECLAM_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() AS db;

---

## Paso 2: Generar Reclamaciones Sintéticas

### Qué Vamos a Crear

**1.000 reclamaciones** bancarias con textos realistas en español.

In [ ]:
-- Tabla de reclamaciones
CREATE OR REPLACE TABLE RECLAMACIONES (
    reclamacion_id VARCHAR(10) PRIMARY KEY,
    cliente_id VARCHAR(10),
    fecha DATE,
    canal VARCHAR(20),
    texto TEXT,
    categoria_real VARCHAR(30)
);

INSERT INTO RECLAMACIONES
SELECT
    'REC' || LPAD(SEQ4()::STRING, 5, '0'),
    'CLI' || LPAD(UNIFORM(1, 5000, RANDOM())::STRING, 5, '0'),
    DATEADD('day', -UNIFORM(0, 180, RANDOM()), CURRENT_DATE()),
    ARRAY_CONSTRUCT('Email','App','Oficina','Teléfono','Web')[UNIFORM(0,4,RANDOM())]::VARCHAR,
    CASE UNIFORM(1, 10, RANDOM())
        WHEN 1 THEN 'Me han cobrado una comisión de mantenimiento que nunca me informaron. Llevo años como cliente y este trato es inaceptable. Exijo la devolución inmediata.'
        WHEN 2 THEN 'La app del banco no funciona desde hace tres días. No puedo hacer transferencias ni consultar mi saldo. Esto es un servicio pésimo para un banco que se dice digital.'
        WHEN 3 THEN 'He detectado un cargo de 500€ en mi tarjeta que no reconozco. Puede ser fraude. Necesito que bloqueen la tarjeta y me devuelvan el dinero urgentemente.'
        WHEN 4 THEN 'El cajero de la oficina de Gran Vía se ha tragado mi tarjeta y nadie me da solución. Llevo esperando 45 minutos y la atención es lamentable.'
        WHEN 5 THEN 'Las comisiones por transferencia internacional son abusivas. En otros bancos es gratis y aquí me cobran 15€. Estoy considerando cambiar de banco.'
        WHEN 6 THEN 'Solicité una hipoteca hace dos meses y nadie me ha contactado. La comunicación es nula y no hay transparencia en el proceso.'
        WHEN 7 THEN 'La web del banco está muy lenta y se cae constantemente. No puedo acceder a mis extractos ni gestionar mis productos. Muy decepcionante.'
        WHEN 8 THEN 'Me han aplicado un tipo de interés diferente al pactado en mi préstamo personal. Esto es una falta de transparencia grave.'
        WHEN 9 THEN 'Agradezco la rapidez en resolver mi incidencia anterior. Sin embargo, esta vez el problema con las domiciliaciones sigue sin resolverse.'
        ELSE 'El servicio de atención telefónica es horrible. He estado 30 minutos en espera y me han pasado por tres departamentos sin resolver nada.'
    END,
    CASE UNIFORM(1, 10, RANDOM())
        WHEN 1 THEN 'Comisiones' WHEN 2 THEN 'Digital' WHEN 3 THEN 'Fraude'
        WHEN 4 THEN 'Servicio' WHEN 5 THEN 'Comisiones' WHEN 6 THEN 'Otros'
        WHEN 7 THEN 'Digital' WHEN 8 THEN 'Comisiones' WHEN 9 THEN 'Servicio'
        ELSE 'Servicio'
    END
FROM TABLE(GENERATOR(ROWCOUNT => 1000));

SELECT categoria_real, COUNT(*) FROM RECLAMACIONES GROUP BY 1 ORDER BY 2 DESC;

---

## Paso 3: Análisis de Sentimiento

### `CORTEX.SENTIMENT()`

Devuelve un score de sentimiento entre -1 (muy negativo) y +1 (muy positivo).

### Interpretación

| Score | Significado | Acción |
|---|---|---|
| < -0.5 | Muy negativo | Escalado urgente |
| -0.5 a -0.2 | Negativo | Atención prioritaria |
| -0.2 a +0.2 | Neutral | Proceso normal |
| > +0.2 | Positivo | Seguimiento estándar |

In [ ]:
-- Analizar sentimiento y clasificar con IA
CREATE OR REPLACE TABLE ANALISIS_RECLAMACIONES AS
SELECT
    reclamacion_id,
    cliente_id,
    fecha,
    canal,
    texto,
    categoria_real,
    SNOWFLAKE.CORTEX.SENTIMENT(texto) AS score_sentimiento,
    CASE
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(texto) < -0.5 THEN 'Muy Negativo'
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(texto) < -0.2 THEN 'Negativo'
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(texto) < 0.2 THEN 'Neutral'
        ELSE 'Positivo'
    END AS nivel_sentimiento,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large',
        'Clasifica esta reclamación bancaria en UNA de estas categorías: Comisiones, Servicio, Digital, Fraude, Otros. Responde SOLO con la categoría.\n\nTexto: ' || texto
    ) AS categoria_ia
FROM RECLAMACIONES;

SELECT nivel_sentimiento, COUNT(*), ROUND(AVG(score_sentimiento), 3) AS score_medio
FROM ANALISIS_RECLAMACIONES GROUP BY 1 ORDER BY score_medio;

---

## Paso 4: Priorización y Respuestas Automáticas

### Priorización por Urgencia

Combinamos sentimiento + categoría para determinar urgencia:
- Fraude + sentimiento negativo → URGENTE
- Sentimiento muy negativo → ALTA
- Resto → según categoría

In [ ]:
-- Priorización y respuestas automáticas
CREATE OR REPLACE TABLE RECLAMACIONES_PRIORIZADAS AS
SELECT
    reclamacion_id, cliente_id, fecha, canal, texto,
    score_sentimiento, nivel_sentimiento, categoria_ia,
    CASE
        WHEN TRIM(categoria_ia) ILIKE '%Fraude%' AND score_sentimiento < -0.3 THEN 'URGENTE'
        WHEN score_sentimiento < -0.5 THEN 'ALTA'
        WHEN score_sentimiento < -0.2 THEN 'MEDIA'
        ELSE 'BAJA'
    END AS prioridad,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large',
        'Genera un borrador de respuesta profesional (3 líneas) a esta reclamación bancaria. Sé empático y ofrece una solución.\n\nReclamación: ' || texto || '\n\nResponde en español.'
    ) AS respuesta_borrador
FROM ANALISIS_RECLAMACIONES;

SELECT prioridad, COUNT(*) AS n FROM RECLAMACIONES_PRIORIZADAS GROUP BY 1 ORDER BY 1;

---

## Paso 5: Dashboard de Calidad de Atención

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Análisis de Sentimiento — Reclamaciones')
st.markdown('### Panel de Calidad de Atención')

total = session.sql('SELECT COUNT(*) FROM RECLAMACIONES_PRIORIZADAS').collect()[0][0]
urgentes = session.sql("SELECT COUNT(*) FROM RECLAMACIONES_PRIORIZADAS WHERE prioridad='URGENTE'").collect()[0][0]
score_medio = session.sql('SELECT ROUND(AVG(score_sentimiento),3) FROM RECLAMACIONES_PRIORIZADAS').collect()[0][0]

c1, c2, c3 = st.columns(3)
c1.metric('Total Reclamaciones', f'{total:,}')
c2.metric('Urgentes', f'{urgentes:,}')
c3.metric('Sentimiento Medio', f'{score_medio}')

st.markdown('---')
st.subheader('Por Prioridad')
df = session.sql('SELECT prioridad, COUNT(*) AS n FROM RECLAMACIONES_PRIORIZADAS GROUP BY 1 ORDER BY 1').to_pandas()
st.bar_chart(df.set_index('PRIORIDAD'))

st.markdown('---')
st.subheader('Reclamaciones con Respuesta IA')
pri = st.selectbox('Filtrar', ['URGENTE','ALTA','MEDIA','BAJA'])
df_r = session.sql(f"SELECT reclamacion_id, texto, score_sentimiento, prioridad, respuesta_borrador FROM RECLAMACIONES_PRIORIZADAS WHERE prioridad='{pri}' LIMIT 10").to_pandas()
for _, row in df_r.iterrows():
    st.markdown(f"**{row['RECLAMACION_ID']}** (Sentimiento: {row['SCORE_SENTIMIENTO']:.2f})")
    st.info(row['TEXTO'][:200])
    st.success(row['RESPUESTA_BORRADOR'][:300])
    st.markdown('---')

---

## Paso 6: Limpieza (Opcional)

In [ ]:
-- Descomentar para limpiar

-- DROP DATABASE IF EXISTS SENTIMIENTO_RECLAM_DB;